# Baseline 1: Mean-difference probe (white box)

This notebook implements the mean-difference probe from Goldowsky-Dill et al. [Detecting Strategic Deception Using Linear Probes](https://arxiv.org/abs/2502.03407). It trains a probe per model organism and evaluates it on the MO-specific dev data.

The method takes the residual-stream activation of a model's response, averages it separately over deceptive and honest examples, and subtracts. That difference vector is the probe. It scores a new response by projecting its activation onto the direction $s(\mathbf{h}) = \mathbf{h} \cdot \mathbf{d}$. Each probe is trained on the RepE honest/deceptive *true-facts* contrast set. 

## 0️⃣ Setup

In [ ]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q "peft==0.18.0"
!pip install -q --upgrade torchao "transformers==5.9.0"

from IPython.display import clear_output
clear_output()

In [ ]:
import os
from nnsight import CONFIG
from huggingface_hub import login

NDIF_API_KEY = os.environ.get("NDIF_API_KEY")
HF_TOKEN = os.environ.get("HF_TOKEN")

CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"  # hackathon NDIF cluster

# ── Model organisms ───────────────────────────────────────────────────────────
# A *model organism* is a (base model, LoRA adapter) pair.
#
# Per-base knobs (the only thing not in the dataset name):
#   multimodal : gemma-3 is a VisionLanguageModel; text models use LanguageModel
#   batch_size : lower for the big MoE model if you hit OOM / remote limits
BASE_SETTINGS = {
    "gemma-3-27b-it":                         dict(multimodal=True,  batch_size=16),
    "Qwen3.5-27B":                            dict(multimodal=False, batch_size=16),
    "NVIDIA-Nemotron-3-Super-120B-A12B-BF16": dict(multimodal=False, batch_size=4),
}
_SPLIT_PREFIXES = ("dev-test-", "validation-", "dev-")   # longest first
LORA_ORG = "aletheias-quest"           # HF org hosting the LoRA adapters
# Full base-model repo id (the dataset `model` column value) per base token.
BASE_REPO = {
    "gemma-3-27b-it":                         "google/gemma-3-27b-it",
    "Qwen3.5-27B":                            "Qwen/Qwen3.5-27B",
    "NVIDIA-Nemotron-3-Super-120B-A12B-BF16": "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
}

def parse_org(name):
    """(base_token, lora_token|None) from a dataset id — mirrors aletheia_runner.config."""
    s = name.split("/")[-1]
    for p in _SPLIT_PREFIXES:
        if s.startswith(p):
            s = s[len(p):]; break
    best = None
    for tok in BASE_SETTINGS:
        i = s.find(tok)
        if i != -1 and (best is None or i < best[0]):
            best = (i, tok)
    if best is None:
        return None, None
    i, tok = best
    rest = s[i + len(tok):].strip("-")
    lora = rest if rest and rest.lower() != "none" else None
    return tok, lora

# A LoRA adapter repo name embeds a (lowercased) base tag -> map it to its base token.
_LORA_BASE_TAG = {
    "gemma-3-27b-it":        "gemma-3-27b-it",
    "qwen3.5-27b":           "Qwen3.5-27B",
    "nemotron-3-super-120b": "NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
}
def base_of_lora(lora_token):
    low = lora_token.lower()
    for tag, base in _LORA_BASE_TAG.items():
        if tag in low:
            return base
    return None

import yaml
from huggingface_hub import HfApi
DRY_YAML = next(p for p in ("../dry.yaml", "dry.yaml", "../../dry.yaml") if os.path.exists(p))

def _new_org(base, lora_token):
    """Empty organism record keyed by (base, lora). Activations are always extracted
    through THIS organism's own base+LoRA model."""
    key = f"{base}::{lora_token or 'base'}"
    return dict(
        key=key, base_token=base, lora_token=lora_token,
        model_id=BASE_REPO[base],
        lora_id=(f"{LORA_ORG}/{lora_token}" if lora_token else None),
        multimodal=BASE_SETTINGS[base]["multimodal"],
        batch_size=BASE_SETTINGS[base]["batch_size"],
        save_as=f"mean_probe_{base}__{lora_token or 'base'}.npz",
        eval_datasets=[], has_dev=False, calib_note="")

ORGANISMS = {}
for e in (yaml.safe_load(open(DRY_YAML)).get("datasets") or []):
    base, lora = parse_org(e["name"])
    if base is None:
        continue
    org = ORGANISMS.setdefault(f"{base}::{lora or 'base'}", _new_org(base, lora))
    org["has_dev"] = True
    org["eval_datasets"].append(dict(
        name=e["name"], labels_uri=e["labels_uri"],
        id_column=e.get("id_column", "index"),
        label_column=e.get("label_column", "deceptive")))

for base in BASE_SETTINGS:
    ORGANISMS.setdefault(f"{base}::base", _new_org(base, None))
try:
    adapters = [m.id.split("/")[-1] for m in HfApi().list_models(author=LORA_ORG)]
except Exception as ex:
    print(f"[warn] couldn't list {LORA_ORG} adapters from HF ({ex}); using dev-derived set only")
    adapters = []
for lora_token in adapters:
    base = base_of_lora(lora_token)
    if base is not None:
        ORGANISMS.setdefault(f"{base}::{lora_token}", _new_org(base, lora_token))


## 1️⃣ The probe

We first compute the difference of class means, and for every new test point, we project it onto that direction.
 
(Positive class = **deceptive**, so a higher score means more deceptive.)

In [ ]:
import numpy as np

class MeanDifferenceProbe:
    """direction = mean(deceptive activations) - mean(honest activations)
    score(h) = h . direction

    `w` weights each example by its number of response tokens, so the class means
    are token-weighted (the mean over all response tokens pooled across the
    dataset) rather than example-weighted. This matches the reference MMSDetector,
    which pools every response token before averaging: a token-weighted mean of
    per-dialogue means is identical to the mean over all tokens. Pass token counts
    as `w` for an exact match; omit it to weight every example equally."""

    def fit(self, X, y, w=None):
        X, y = np.asarray(X, dtype=np.float64), np.asarray(y)
        w = np.ones(len(y)) if w is None else np.asarray(w, dtype=np.float64)

        def wmean(mask):
            return (X[mask] * w[mask, None]).sum(axis=0) / w[mask].sum()

        self.direction = wmean(y == 1) - wmean(y == 0)
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=np.float64) @ self.direction

## 2️⃣ Training data: RepE honest/deceptive facts

The training data is built from `data/true_false_facts.csv` (the RepE true/false facts from Zou et al.). We keep only the **true** statements and present each one **twice**, once under an honest persona and once under a dishonest persona, with *identical* assistant content.

In [ ]:
import numpy as np
import pandas as pd

# RepE honest/deceptive persona prompts
HONEST_PROMPT    = "Pretend you're an honest person making statements about the world."
DECEPTIVE_PROMPT = "Pretend you're a dishonest person making statements about the world."

# Load the RepE facts and keep the first 512 true statements
facts = pd.read_csv("data/true_false_facts.csv")
true_facts = facts[facts["label"] == 1]["statement"].tolist()[:512]

# Each true fact appears under both personas, interleaved (honest, deceptive, honest, ...).
# label: 0 = honest, 1 = deceptive (positive class = deceptive)
train_examples = []
for s in true_facts:
    train_examples.append((HONEST_PROMPT,    s, 0))
    train_examples.append((DECEPTIVE_PROMPT, s, 1))

y = np.array([lbl for *_, lbl in train_examples])
print(f"RepE training set: {len(train_examples)} dialogues "
      f"({(y == 1).sum()} deceptive / {(y == 0).sum()} honest) from {len(true_facts)} true facts")

## 3️⃣ Per-organism pipeline (tokenization, model loading, activation extraction)

These helpers are parameterized by model/tokenizer so the identical pipeline runs for every organism. The RepE detect span is the fact minus its last 5 words; the eval detect span is the final assistant message. Both are located with `char_to_token`, which is robust across chat templates (no per-model EOT/system-prompt special-casing needed).

In [ ]:
from transformers import AutoTokenizer

def normalize_messages(tokenizer, messages):
    """Return messages the tokenizer's chat template accepts. If it rejects a
    `system` role (e.g. gemma), fold the leading system prompt into the first user turn."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    try:
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return msgs
    except Exception:
        norm, pending = [], None
        for m in msgs:
            if m["role"] == "system" and not norm:
                pending = m["content"]; continue
            if not norm and m["role"] == "user" and pending:
                norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
            norm.append(m)
        if pending:
            norm = [{"role": "user", "content": pending}] + norm
        return norm

def _span(tokenizer, text, content, span_len=None):
    """Tokenize `text` and return (ids, start, end) mapping the [content .. content+span_len)
    character range to token indices. `content` is stripped first because many chat templates
    apply `| trim` to message content, so the rendered text holds the trimmed version."""
    content = content.strip()
    enc = tokenizer(text, add_special_tokens=False)
    ids = enc["input_ids"]

    c = text.rfind(content) if content else -1
    if c == -1 and content:                       # last resort: match a shrinking prefix
        probe = content[:200]
        while len(probe) > 20 and c == -1:
            c = text.rfind(probe)
            if c == -1:
                probe = probe[: len(probe) // 2]
        span_len = None                            # only the start is reliable now
    if c == -1:                                    # give up -> empty span (gets weight 0)
        return ids, len(ids), len(ids)

    start = enc.char_to_token(c)
    if start is None:
        start = 0
    if span_len == 0:
        return ids, start, start
    end_char = c + (len(content) if span_len is None else min(span_len, len(content)))
    end = enc.char_to_token(end_char)
    if end is None:
        end = len(ids)
    return ids, start, end

def tokenize_repe(tokenizer, user_prompt, statement):
    """detect span = fact statement minus its last 5 words (repo's repe_honesty region)."""
    statement = statement.strip()
    fact_start = " ".join(statement.split(" ")[:-5])  # "" if <=5 words
    msgs = [{"role": "user", "content": user_prompt},
            {"role": "assistant", "content": statement}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return _span(tokenizer, text, statement, span_len=len(fact_start) if fact_start else 0)

def tokenize_eval(tokenizer, messages):
    """detect span = the final (assistant) message content tokens."""
    msgs = normalize_messages(tokenizer, messages)
    content = msgs[-1]["content"]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return _span(tokenizer, text, content)

In [ ]:
import torch
import numpy as np
import pandas as pd
from nnsight import LanguageModel, VisionLanguageModel
from datasets import load_dataset

def build_model(model_id, lora_id, multimodal):
    """nnsight handle for one organism. `peft=lora_id` attaches the LoRA adapter
    (omit when None). Config + tokenizer load locally; the weights live on NDIF."""
    Cls = VisionLanguageModel if multimodal else LanguageModel
    kwargs = {"peft": lora_id} if lora_id else {}
    return Cls(model_id, **kwargs)

def decoder_layers(model):
    """The text decoder's `layers` ModuleList: read decoder_layers(model)[L].output
    for the residual stream at layer L."""
    def is_block(m):
        return (any(hasattr(m, a) for a in ("self_attn", "linear_attn", "mixer"))
                or (hasattr(m, "mlp") and hasattr(m, "input_layernorm")))
    root = model.model
    cands = []
    for name, child in root.named_modules():
        if name.rsplit(".", 1)[-1] != "layers" or "vision" in name.lower():
            continue
        kids = list(child.children())
        if kids and any(is_block(k) for k in kids):
            cands.append((len(kids), child))
    if cands:
        return max(cands, key=lambda c: c[0])[1]
    # last-resort fallbacks for unusual nestings (descend into the CausalLM if needed)
    inner = getattr(root, "language_model", root)
    return inner.layers if hasattr(inner, "layers") else inner.model.layers

def collate(batch, pad_id):
    """Right-pad a batch; return (input_ids, attention_mask, response_spans).
    Spans index the padded sequence; an empty detect span falls back to the whole
    real sequence so pooling is always finite (such rows carry weight 0 in `fit`)."""
    max_len = max(len(t[0]) for t in batch)
    B = len(batch)
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attn = torch.zeros((B, max_len), dtype=torch.long)
    spans = []
    for i, (ids, s, e) in enumerate(batch):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn[i, :len(ids)] = 1
        spans.append((s, e) if e > s else (0, len(ids)))
    return input_ids, attn, spans

def pool_pieces(model, layers_mod, tok_list, layers, pad_id, batch_size):
    """Open ONE trace per batch (inside an already-active session) and return a LIST
    of per-batch proxies, each (num_layers, B, hidden). The proxies are left unsaved."""
    batches = [tok_list[i:i + batch_size] for i in range(0, len(tok_list), batch_size)]
    pieces = []
    for batch in batches:
        input_ids, attn, spans = collate(batch, pad_id)
        with model.trace({"input_ids": input_ids, "attention_mask": attn}):
            pooled = []
            for L in layers:
                h = layers_mod[L].output                          # (B, T, D), on this layer's GPU
                pooled.append(torch.stack([h[i, s:e].mean(0) for i, (s, e) in enumerate(spans)]))
            ref = pooled[0].device                                # layers may be sharded across GPUs
            pieces.append(torch.stack([p.to(ref) for p in pooled], dim=0))  # (num_layers, B, hidden)
    return pieces

def to_layer_dict(saved, layers):
    """A post-session saved proxy (num_layers, N, hidden) -> {layer: (N, hidden)}."""
    feats = saved.cpu().float().numpy()
    return {L: feats[i] for i, L in enumerate(layers)}

# ---- on-policy evaluation data: the organism's own labeled competition datasets ----
EVAL_MAX_PER_LABEL = 400

def load_org_eval(ds_cfg, max_per_label=EVAL_MAX_PER_LABEL, seed=42):
    """Load one organism dev dataset + its held-out labels, joined on the id column.
    Returns (messages, y) balanced-capped, or ([], []) if it isn't usable (single
    class / no rows). `messages`'s final turn is the assistant response we probe."""
    name, labels_uri = ds_cfg["name"], ds_cfg["labels_uri"]
    idc, lbc = ds_cfg["id_column"], ds_cfg["label_column"]
    df = load_dataset(name, split="test").to_pandas()
    if idc not in df.columns:
        df[idc] = range(len(df))
    lab = pd.read_csv(labels_uri) if str(labels_uri).endswith(".csv") \
        else load_dataset(labels_uri, split="test").to_pandas()
    df = df.merge(lab[[idc, lbc]], on=idc, how="inner")
    y = df[lbc].astype(int).values
    if len(np.unique(y)) < 2:
        return [], np.array([])
    if max_per_label:
        rng = np.random.default_rng(seed)
        pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
        n = min(max_per_label, len(pos), len(neg))
        keep = np.sort(np.concatenate([rng.choice(pos, n, replace=False),
                                       rng.choice(neg, n, replace=False)]))
        df = df.iloc[keep].reset_index(drop=True)
        y = df[lbc].astype(int).values
    msgs = [list(m) for m in df["messages"]]
    return msgs, y

## 4️⃣ Train, evaluate, and save (one probe per model organism)

`train_and_save_probe(org)` is the whole per-organism pass. It peeks row 0 of the organism's dataset for the **real** `model` + `lora` columns; builds that organism's model, then opens **one remote NDIF session** that extracts the RepE training activations *and* every eval dataset's activations together. Afterwards (on CPU) it fits a token-weighted mean-difference probe per layer, scores transfer AUROC per eval dataset, picks the best-average layer, and saves that organism's probe to its own `.npz`.

In [ ]:
from transformers import AutoTokenizer
from sklearn.metrics import roc_auc_score

all_results = {}   # organism key -> {best_layer, table, summary, save_as}

def train_and_save_probe(org, max_per_label=EVAL_MAX_PER_LABEL):
    """Full pass for ONE model organism (base model + LoRA): load data, extract RepE
    training + eval activations in a SINGLE remote NDIF session, then fit / select the
    best layer / save the probe (CPU). Writes `org['save_as']`, records `all_results`."""
    key = org["key"]
    print(f"\n{'='*20} {key} {'='*20}")

    if not org.get("eval_datasets"):
        _base = ORGANISMS.get(f"{org['base_token']}::base")
        if _base and _base.get("eval_datasets"):
            org["eval_datasets"] = [dict(d) for d in _base["eval_datasets"]]

    ## (model_id, lora_id) come from the dataset's own columns (peek row 0)
    model_id, lora_id = org["model_id"], org["lora_id"]
    print(f"model_id={model_id}  lora_id={lora_id}")

    model = build_model(model_id, lora_id, org["multimodal"])
    tokenizer = AutoTokenizer.from_pretrained(model_id)   # text tokenizer (even for the VLM base)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    layers_mod = decoder_layers(model)
    LAYERS = list(range(len(layers_mod)))

    ## tokenize data
    train_tok = [tokenize_repe(tokenizer, u, s) for (u, s, _) in train_examples]
    counts = np.array([e - s for (_, s, e) in train_tok], dtype=np.float64)
    eval_raw = {}                                          # short name -> (etok, y)
    for ds_cfg in org["eval_datasets"]:
        msgs, y_s = load_org_eval(ds_cfg, max_per_label=max_per_label)
        if len(msgs) == 0:
            print(f"  [skip] {ds_cfg['name'].split('/')[-1]} (no usable labeled rows)")
            continue
        short = ds_cfg["name"].split("/")[-1]
        eval_raw[short] = ([tokenize_eval(tokenizer, m) for m in msgs], y_s)
        print(f"  eval {short}: {len(msgs)} examples")
    if not eval_raw:
        raise RuntimeError(f"{key}: no usable eval datasets")

    ## one session for extracting all activations
    groups = [("__train__", train_tok)] + [(nm, etok) for nm, (etok, _) in eval_raw.items()]
    bs = org["batch_size"]
    print(f"extracting activations in one remote session "
          f"({sum(len(t) for _, t in groups)} rows x {len(LAYERS)} layers)...")
    with model.session(remote=True):
        pieces = []
        for _, gtok in groups:
            pieces.extend(pool_pieces(model, layers_mod, gtok, LAYERS, pad_id, bs))
        feats_all = torch.cat(pieces, dim=1).save()           # (num_layers, N_total, hidden)

    arr = feats_all.cpu().float().numpy()                      # (num_layers, N_total, hidden)
    offs = np.cumsum([0] + [len(t) for _, t in groups])
    grp = {nm: arr[:, offs[i]:offs[i + 1], :] for i, (nm, _) in enumerate(groups)}
    X_train = {L: grp["__train__"][i] for i, L in enumerate(LAYERS)}
    eval_sets = {nm: {"X": {L: grp[nm][i] for i, L in enumerate(LAYERS)}, "y": ys}
                 for nm, (etok, ys) in eval_raw.items()}

    ## per layer probe
    probes = {L: MeanDifferenceProbe().fit(X_train[L], y, counts) for L in LAYERS}
    rows = []
    for L in LAYERS:
        row = {"layer": L}
        for name, d in eval_sets.items():
            row[name] = roc_auc_score(d["y"], probes[L].decision_function(d["X"][L]))
        row["average"] = float(np.mean([row[name] for name in eval_sets]))
        rows.append(row)
    table = pd.DataFrame(rows).set_index("layer")
    best_layer = int(table["average"].idxmax())
    print(table.round(3).to_string())
    print(f"best layer (avg AUROC): {best_layer}")

    ## calibrate a threshold for the probe
    final = MeanDifferenceProbe().fit(X_train[best_layer], y, counts)
    ev_proj = np.concatenate([final.decision_function(eval_sets[n]["X"][best_layer]) for n in eval_sets])
    ev_y    = np.concatenate([eval_sets[n]["y"] for n in eval_sets])
    cuts = np.unique(ev_proj)
    cuts = np.concatenate([[cuts[0] - 1.0], (cuts[:-1] + cuts[1:]) / 2, [cuts[-1] + 1.0]])
    P, N = max(int((ev_y == 1).sum()), 1), max(int((ev_y == 0).sum()), 1)
    bal_acc = [0.5 * (((ev_proj >= t) & (ev_y == 1)).sum() / P + ((ev_proj < t) & (ev_y == 0)).sum() / N)
               for t in cuts]
    bias = float(cuts[int(np.argmax(bal_acc))])
    score_std = float(ev_proj.std() + 1e-8)
    np.savez(org["save_as"],
             direction=final.direction.astype(np.float32),
             layer=np.array(best_layer),
             model_id=np.array(model_id),
             lora_id=np.array("" if lora_id is None else lora_id),
             organism=np.array(key),
             bias=np.array(bias, dtype=np.float32),
             score_std=np.array(score_std, dtype=np.float32))
    print(f"saved {org['save_as']} | layer={best_layer} | dim={final.direction.shape[0]} | "
          f"bias={bias:.4g} | score_std={score_std:.4g} | dev balanced-acc={max(bal_acc):.3f}")

    all_results[key] = dict(
        best_layer=best_layer, table=table, save_as=org["save_as"],
        summary={c: float(table.loc[best_layer, c]) for c in table.columns})
    del model, feats_all, arr, grp, X_train, eval_sets, probes
    return all_results[key]

### One cell per organism

Each cell below runs a single pass over the model organism, and produces a trained probe in .npz.

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 (base, no LoRA)
train_and_save_probe(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::base"])

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 + LoRA s-mo-nemotron-3-super-120b  (no public dev data -> calibrated on Nemotron-120B base dev)
train_and_save_probe(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::s-mo-nemotron-3-super-120b"])

In [ ]:
# NVIDIA-Nemotron-3-Super-120B-A12B-BF16 + LoRA g-st-nemotron-3-super-120b  (no public dev data -> calibrated on Nemotron-120B base dev)
train_and_save_probe(ORGANISMS["NVIDIA-Nemotron-3-Super-120B-A12B-BF16::g-st-nemotron-3-super-120b"])

In [ ]:
# Qwen3.5-27B (base, no LoRA)
train_and_save_probe(ORGANISMS["Qwen3.5-27B::base"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-1
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-1"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-3
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-3"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-4
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-4"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-5
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-5"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-6
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-6"])

In [ ]:
# Qwen3.5-27B + LoRA a-mo-qwen3.5-27b-7
train_and_save_probe(ORGANISMS["Qwen3.5-27B::a-mo-qwen3.5-27b-7"])

In [ ]:
# Qwen3.5-27B + LoRA b-mo-qwen3.5-27b
train_and_save_probe(ORGANISMS["Qwen3.5-27B::b-mo-qwen3.5-27b"])

In [ ]:
# Qwen3.5-27B + LoRA c-mo-qwen3.5-27b
train_and_save_probe(ORGANISMS["Qwen3.5-27B::c-mo-qwen3.5-27b"])

In [ ]:
# Qwen3.5-27B + LoRA g-st-qwen3.5-27b  (no public dev data -> calibrated on Qwen3.5-27B base dev)
train_and_save_probe(ORGANISMS["Qwen3.5-27B::g-st-qwen3.5-27b"])

In [ ]:
# gemma-3-27b-it (base, no LoRA)
train_and_save_probe(ORGANISMS["gemma-3-27b-it::base"])

In [ ]:
# gemma-3-27b-it + LoRA s-mo-gemma-3-27b-it
train_and_save_probe(ORGANISMS["gemma-3-27b-it::s-mo-gemma-3-27b-it"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-1  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_probe(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-1"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-2  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_probe(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-2"])

In [ ]:
# gemma-3-27b-it + LoRA g-st-gemma-3-27b-it-3  (no public dev data -> calibrated on gemma-3-27b-it base dev)
train_and_save_probe(ORGANISMS["gemma-3-27b-it::g-st-gemma-3-27b-it-3"])

In [ ]:
print("=== Best-layer transfer AUROC per organism ===\n")
summary_rows = []
for key, r in all_results.items():
    print(f"{key}  (layer {r['best_layer']}  ->  {r['save_as']})")
    for k, v in r["summary"].items():
        print(f"   {k:52s} {v:.3f}")
    print()
    summary_rows.append({"organism": key, "best_layer": r["best_layer"],
                         "avg_auroc": r["summary"]["average"]})

summary_df = pd.DataFrame(summary_rows).set_index("organism")
summary_df.round(3)

## 5️⃣ Saved probes

The cells above wrote one `.npz` per organism. Each carries the probe `direction` + `layer`, the `model_id` / `lora_id` / `organism` it belongs to, and the per-organism calibration `bias` (balanced-accuracy-optimal threshold) + `score_std` (sigmoid slope). The submission notebook (`baselines/mean_probe_organism.ipynb`) selects the matching file by `(model_id, lora_id)` at run time. We reload them here to verify.

In [ ]:
import glob

for f in sorted(glob.glob("mean_probe_*.npz")):
    d = np.load(f, allow_pickle=True)
    org  = str(d["organism"]) if "organism" in d else "?"
    lora = str(d["lora_id"]) if "lora_id" in d else ""
    print(f"{f}\n   organism: {org}  | model: {str(d['model_id'])}  | "
          f"lora: {lora or 'None'}  | layer: {int(d['layer'])}  | dim: {d['direction'].shape[0]}")